# M0.2 · The loop

**Module 0 — the shared core → The Shared Core**  ·  *Both directions*

---

**Risk.** A harness with no verifier converges on plausible-looking garbage.

**Control.** Every loop has a context, a toolset, a verifier and a budget — the verifier is the security control.

**This lab.** Build the minimum loop and prove the verifier is the security control.

| | |
|---|---|
| Open-source tooling | Python, Ollama |
| Open-weight models | Kimi K2, GLM-4.6 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("M0.2"))

The loop is four moves — plan, act, verify, stop. Everything that makes it safe or unsafe lives in the last two.

Run one broken proposal past three different verifiers. The proposal never changes; only what the loop believes about it does.

In [ ]:
from cybercommons import loop

BROKEN  = "def add(a, b): return a - b"
CORRECT = "def add(a, b): return a + b"

results = loop.compare_verifiers(
    lambda: loop.FakeModel([BROKEN]),
    {"oracle (deterministic)": loop.oracle(CORRECT),
     "unit test":              loop.unit_test(lambda s: "a + b" in s, "checks the operator"),
     "llm-judge (self-grading)": loop.llm_judge(),
     "none":                   loop.no_verifier()},
    max_steps=3)

for name, r in results.items():
    print(f"{name:26s} succeeded={str(r['succeeded']):5s}  stopped_by={r['stopped_by']}")

The code is wrong in every row. Two verifiers say so; one declares success; one never decides at all and is stopped by the budget.\n\nNow look at the trace the self-grading loop produces — because this is what an operator would actually see.

In [ ]:
trace = loop.run(loop.FakeModel([BROKEN]), loop.llm_judge(),
                 goal="fix the add function", max_steps=3)
print(trace.table())
print("\nThe trace is clean. The code is broken. Nothing in the trace says so.")

### Expect

`oracle` and `unit test` both fail and stop on the step budget. `llm-judge` returns succeeded=True on the first step against code that computes subtraction. `none` runs the full budget. The final trace prints as a tidy success.

### Your turn

Write a verifier that would catch this without knowing the expected answer in advance — e.g. one that executes the function against a property (`add(2,2) == 4`). That is the difference between a judge and an oracle.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/M0.2.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*